### script for displaying the GSAT, ARC, and ARC/GSAT ratio curves, 
[SSP1-1.9; 1-2.6] & Shared x axes, with extended Y axes[right Y axis: Arctic warming fraction(top); left Y axis: Arctic Ocean Temperature anomalies(middle); right Y axis: GSAT anomalies (bottom)]

In [ ]:
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from pathlib import Path
import glob
# import proplot as pplt

In [ ]:
import src.SAT_function as data_process

In [ ]:
def running_mean(da, win=5):
    """Centered running mean along time; keeps coords."""
    return da.rolling(time=win, center=True).mean()

def safe_ratio(num, den):
    """ARC/GSAT with protection against near-zero denominator."""
    eps = 1e-6
    return num / xr.where(np.abs(den) < eps, np.nan, den)

In [ ]:
def to_year_dim(ds: xr.Dataset) -> xr.Dataset:
    # If 'year' is a dimension, rename to 'time'
    if "year" in ds.dims:
        ds = ds.rename({'year': 'time'})
        ds = ds.assign_coords(time=ds['time'].astype(int))
    elif "time" in ds.dims:
        # If 'time' is datetime or cftime, convert to integer years
        if hasattr(ds['time'], 'dt'):
            years = ds['time'].dt.year.astype(int)
            ds = ds.assign_coords(time=years)
        else:
            ds = ds.assign_coords(time=ds['time'].astype(int))
    else:
        raise ValueError("Dataset needs a 'time' or 'year' dimension")
    ds = ds.drop_vars("year", errors="ignore")
    if "year" in ds.coords and "year" not in ds.dims:
        ds = ds.reset_coords("year", drop=True)
    return ds

### input the Arctic regional mean SAT timeseries

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

LENS_name = ["MIROC", "MIROC-ES2L", "MPI-ESM-LR", "CanESM5"]
base_dir = "/work/mh0033/m301036/Land_surf_temp/Observational_constraints/Regional_mechanism_scatter/data/regional_mean"
scenarios = ["ssp119", "ssp126"]

ARC_LEs_datasets = {}

for scenario in scenarios:
    for model in LENS_name:
        print(f"Processing model: {model} for scenario: {scenario}")
        file_path = os.path.join(base_dir, f"{model}_AR6_RegionalMeans_1850_2100.nc")
        if not os.path.exists(file_path):
            print(f"  -> File not found: {file_path}")
            continue

        try:
            ds = xr.open_dataset(file_path)
        except Exception as e:
            print(f"  -> Error opening {file_path}: {e}")
            continue

        # detect ARO variable name
        var_candidates = ["ARO", "ARO SAT anomaly", "ARO_SAT_anomaly", "ARO_SAT"]
        varname = None
        for v in var_candidates:
            if v in ds.data_vars:
                varname = v
                break
        if varname is None:
            print(f"  -> No ARO variable found in {file_path}. Available vars: {list(ds.data_vars)}")
            continue

        # normalize time coordinate to integer years using your helper
        try:
            ds = to_year_dim(ds)   # assumes to_year_dim is defined in the notebook
        except Exception as e:
            print(f"  -> to_year_dim failed for {file_path}: {e}")
            # still continue but don't fail
        # check scenario coordinate values
        if "scenario" in ds.coords:
            scen_vals = np.array(ds["scenario"].values, dtype=object)
            # try direct match
            if np.any(scen_vals == scenario):
                sel_scen = scenario
            else:
                # try case-insensitive match
                scen_str = np.array([str(s).lower() for s in scen_vals])
                matches = np.where(scen_str == scenario.lower())[0]
                if matches.size:
                    sel_scen = scen_vals[matches[0]]
                    print(f"  -> using scenario coord value {sel_scen} for requested {scenario}")
                else:
                    print(f"  -> scenario '{scenario}' not in file. Available: {scen_vals}. Skipping.")
                    continue
            try:
                da = ds[varname].sel(scenario=sel_scen)
            except Exception as e:
                print(f"  -> selection by scenario failed: {e}")
                continue
        else:
            # no scenario coord: assume file already contains the requested scenario
            da = ds[varname]

        # squeeze model dim if present
        if "model" in da.dims:
            try:
                da = da.squeeze("model", drop=True)
            except Exception:
                # if squeeze fails leave as is
                pass

        # ensure member dim exists; if missing, create a single-member dim using filename
        if "member" not in da.dims:
            da = da.expand_dims("member")
            da = da.assign_coords(member=[f"{model}"])

        # ensure time coordinate name is 'time' and sorted
        if "time" not in da.dims and "year" in da.dims:
            da = da.rename({"year": "time"})
        if "time" in da.coords and np.issubdtype(da["time"].dtype, np.datetime64):
            # convert to integer years
            try:
                years = da["time"].dt.year.values
                da = da.assign_coords(time=years)
            except Exception:
                pass

        da = da.sortby("time")
        # save
        ARC_LEs_datasets[(model, scenario)] = da
        print(f"  -> stored with dims: {da.dims}")

# quick verification print
for (model, scenario), da in ARC_LEs_datasets.items():
    print(f"Stored: {model}, {scenario} -> dims: {da.dims}, times: {da['time'].shape}")


In [ ]:
ARC_ENS_MEAN = {}

for model in LENS_name:
    ARC_ENS_MEAN[model] = {}
    for scenario in scenarios:
        key = (model, scenario)
        if key in ARC_LEs_datasets:
            ARC_ENS_MEAN[model][scenario] = ARC_LEs_datasets[key].mean(dim='member')
        else:
            print(f"Warning: No data for {key}, skipping.")

ARC_ENS_MEAN

In [ ]:
ARC_MMM = {}

for scenario in scenarios:
    ARC_MMM[scenario] = xr.concat([ARC_ENS_MEAN[model][scenario] for model in LENS_name], dim='model').mean(dim='model')

ARC_MMM

### GSAT TS

In [ ]:
import os
import xarray as xr

LENS_name = ["MIROC", "MIROC-ES2L", "MPI-ESM-LR", "CanESM5"]
base_dir = "/work/mh0033/m301036/Land_surf_temp/Observational_constraints/Timeseries/deseasonalized_output/"
scenarios = ["ssp119", "ssp126"]
members = {
    "CanESM5":      [f"r{i}i1p1f1" for i in range(1, 26)] + [f"r{i}i1p2f1" for i in range(1, 26)],
    "MPI-ESM-LR":   [f"r{i}i1p1f1" for i in range(1, 51)],
    "MIROC":        [f"r{i}i1p1f1" for i in range(1, 51)],
    "MRI-ESM2-0":   [f"r{i}i1p1f1" for i in range(1, 21)],
    "IPSL-CM6A-LR": [f"r{i}i1p1f1" for i in range(1, 32)],
    "MIROC-ES2L":   [f"r{i}i1p1f2" for i in range(1, 31)],
    "GISS-E2-1-G":  [f"r{i}i1p1f2" for i in range(1, 11)] + [f"r{i}i1p5f1" for i in range(1, 11)] + [f"r{i}i1p3f1" for i in range(1, 11)],
    "CNRM-ESM2-1":  [f"r{i}i1p1f2" for i in range(1, 11)],
    "UKESM1-0-LL":  [f"r{i}i1p1f2" for i in range(1, 21)],
}

model_datasets = {}

for scenario in scenarios:
    scenario_dir = os.path.join(base_dir, scenario)
    for model in LENS_name:
        print(f"Processing model: {model} for scenario: {scenario}")
        model_dir = os.path.join(scenario_dir, model)
        if model not in members:
            # print(f"No members defined for model {model}. Skipping.")
            continue
        print(f"Model Directory: {model_dir}")

        member_datasets = []
        for member in members[model]:
            file_pattern = f"{model_dir}/{model}_{scenario}_{member}_1850-2100_GMSAT_anomalies_wrt_1850-1900.nc"
            files = sorted(os.popen(f'ls {file_pattern} 2>/dev/null').read().split())
            if not files:
                continue
            try:
                ds = xr.open_dataset(files[0])
                # Rename variable to 'tas' if needed
                if 'tas_ano' in ds.data_vars and 'tas' not in ds.data_vars:
                    ds = ds.rename({'tas_ano': 'tas'})
                # Always convert to integer year 'time' coordinate for consistency
                ds = to_year_dim(ds)
                ds = ds.expand_dims(dim="member").assign_coords(member=[member])
                member_datasets.append(ds)
            except Exception as e:
                print(f"Error loading {model}-{member}: {e}")
                continue

        if member_datasets:
            try:
                model_ds = xr.concat(member_datasets, dim="member", coords="minimal")
                model_ds.attrs["model_name"] = model
                model_ds.attrs["scenario"] = scenario
                model_datasets[(model, scenario)] = model_ds
            except Exception as e:
                print(f"Error concatenating members for {model} {scenario}: {e}")
        else:
            print(f"No valid data found for model {model} in scenario {scenario}.")

for (model, scenario), ds in model_datasets.items():
    print(f"Model: {model}, Scenario: {scenario}, Dataset Shape: {ds.dims}")

In [ ]:
# calculate the mean of each model 
model_mean = {}

for model in LENS_name:
    model_mean[model] = {}
    for scenario in scenarios:
        key = (model, scenario)
        if key in model_datasets:
            model_mean[model][scenario] = model_datasets[key].mean(dim='member')
        else:
            print(f"Warning: No data for {key}, skipping.")

model_mean

In [ ]:
model_mean['MIROC-ES2L']['ssp119']

In [ ]:
# saving each model mean to a netcdf file
output_combined_dir = "/work/mh0033/m301036/Land_surf_temp/Observational_constraints/Timeseries/deseasonalized_output/LENS_combined"
os.makedirs(output_combined_dir, exist_ok=True)

for model in LENS_name:
    for scenario in scenarios:
        output_file = f"{output_combined_dir}/{model}_{scenario}_GMSAT_timeseries_mean.nc"
        model_mean[model][scenario].to_netcdf(output_file)
        print(f"Saved {model} dataset for scenario {scenario} to {output_file}")

In [ ]:
# calculate the ensemble mean of each scenario to compare with the observation
ensemble_mean = {}

for scenario in scenarios:
    ensemble_mean[scenario] = xr.concat([model_mean[model][scenario] for model in LENS_name], dim='model').mean(dim='model')
    
ensemble_mean

In [ ]:
# saving the ensemble mean to a netcdf file
# for scenario in scenarios:
#     output_file = f"{output_combined_dir}/ensemble_mean_{scenario}_GMSAT_timeseries_mean.nc"
#     ds = ensemble_mean[scenario]
#     # Standardize time coordinate to integer years if needed
#     if np.issubdtype(ds['time'].dtype, np.datetime64):
#         ds = ds.assign_coords(time=ds['time'].dt.year.astype(int))
#     elif ds['time'].dtype == object:
#         # If time is mixed types, try to extract year for each entry
#         years = []
#         for t in ds['time'].values:
#             if hasattr(t, 'year'):
#                 years.append(int(t.year))
#             else:
#                 years.append(int(t))
#         ds = ds.assign_coords(time=("time", years))
#     ds.to_netcdf(output_file)
#     print(f"Saved ensemble dataset for scenario {scenario} to {output_file}")

### Calculate the Arctic Amplification Ratio:
delta(SAT_Arctic)/delta(SAT_Global)

In [ ]:
model_datasets["MIROC","ssp119"]['tas']

In [ ]:
# Arctic/GSAT Ratios
def safe_ratio(num, den, eps=1e-6):
    den2 = xr.where(np.abs(den) < eps, np.nan, den)
    return (num / den2).rename('AA')
def ensure_dims(da, member_name='member'):
    # unify member dim name
    if member_name not in da.dims:
        alt = [d for d in da.dims if d.lower() in ('member','realization','ens','ens_member','r')]
        if alt:
            da = da.rename({alt[0]: member_name})
    return da.transpose('time', member_name) if member_name in da.dims else da

ARC_ratios_datasets = {}
for scenario in scenarios:
    for model in LENS_name:
        ARC = ARC_LEs_datasets.get((model, scenario))
        GSAT = model_datasets.get((model, scenario))
        if ARC is None or GSAT is None or 'tas' not in GSAT:
            print(f"-> Missing data for {model} {scenario}. Skip.")
            continue

        ARC  = ensure_dims(ARC)                # (time, member)
        GSAT = ensure_dims(GSAT['tas'])        # (time, member) GSAT already aggregated

        # (optional) enforce anomalies here if not already anomalies
        # ARC, GSAT = to_anom(ARC), to_anom(GSAT)

        # align on both coords
        ARC, GSAT = xr.align(ARC, GSAT, join='inner', copy=False)

        # AA per member
        ratio_da = safe_ratio(ARC, GSAT)

        # clean up impossible times (all-NaN rows)
        ratio_da = ratio_da.dropna('time', how='all')

        ARC_ratios_datasets[(model, scenario)] = ratio_da
        print(f"Stored AA for {model} {scenario}: dims {ratio_da.dims}")

In [ ]:
# quick verification print
for (model, scenario), da in ARC_ratios_datasets.items():
    print(f"Stored Ratio: {model}, {scenario} -> dims: {da.dims}, times: {da['time'].shape}")
ARC_ratios_datasets

In [ ]:
# save the ARO/GSAT ratio datasets to netcdf files
output_ratio_dir = "/work/mh0033/m301036/Land_surf_temp/Observational_constraints/Timeseries/deseasonalized_output/ARO_GSAT_ratios"
os.makedirs(output_ratio_dir, exist_ok=True)

for (model, scenario), da in ARC_ratios_datasets.items():
    output_file = f"{output_ratio_dir}/{model}_{scenario}_ARO_GSAT_ratio_timeseries.nc"
    da.to_netcdf(output_file)
    print(f"Saved ARO/GSAT ratio for {model} scenario {scenario} to {output_file}")

In [ ]:
# calculate the mean of each model ARO/GSAT ratio among the members
ARC_ratio_mean = {}
for (model, scenario), da in ARC_ratios_datasets.items():
    ARC_ratio_mean[(model, scenario)] = da.mean(dim='member')

In [ ]:
ARC_ratio_mean['MIROC-ES2L','ssp119']['time']

In [ ]:
# save the ensemble mean ARO/GSAT ratio datasets to netcdf files
for (model, scenario), da in ARC_ratio_mean.items():
    output_file = f"{output_ratio_dir}/{model}_{scenario}_ARO_GSAT_ratio_timeseries_mean.nc"
    da.to_netcdf(output_file)
    print(f"Saved mean ARO/GSAT ratio for {model} scenario {scenario} to {output_file}")

### checking the GMST timeseries

In [ ]:
#Plotting
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['savefig.transparent'] = True
x = np.arange(1850, 2101, 1)


In [ ]:
from cycler import cycler

model_names = ["MIROC", "MIROC-ES2L", "MPI-ESM-LR", "CanESM5"]

RGB_dict = {
    'CanESM5'     : np.array([148,103,189]) / 255.0,  # purple
    'MPI-ESM-LR'  : np.array([214, 39,  40]) / 255.0, # red
    'MIROC'       : np.array([255,127, 14]) / 255.0,  # orange
    'MIROC-ES2L'  : np.array([ 44,160, 44]) / 255.0,  # green
    'UKESM1-0-LL' : np.array([ 23,190,207]) / 255.0,  # cyan
    'IPSL-CM6A-LR': np.array([ 31,119,180]) / 255.0,  # blue
    'MRI-ESM2-0'  : np.array([255,215,  0]) / 255.0,  # gold
    'CNRM-ESM2-1' : np.array([140, 86, 75]) / 255.0,  # brown
    'GISS-E2-1-G' : np.array([227,119,194]) / 255.0,  # pink/magenta
    'MMLE'         : np.array([  0,  0,  0]) / 255.0,  # black
}

# plt.rc('axes', prop_cycle=cycler(color=[RGB_dict[model] for model in model_names]))

# %%
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerLine2D
from matplotlib.legend import Legend

In [ ]:
min_max_data = {}

for scenario in scenarios:
    for model in LENS_name:
        dataset_key = (model, scenario)
        if dataset_key not in ARC_LEs_datasets:
            print(f"Skipping {dataset_key}, no data available.")
            continue
        
        tas_data = ARC_LEs_datasets[dataset_key]  # Shape (members, years)
        time = ARC_LEs_datasets[dataset_key]['time'].values  # Get time axis
        
        # Compute min/max across ensemble members
        tas_min = tas_data.min(dim="member")  # Shape (times,)
        tas_max = tas_data.max(dim="member")  # Shape (times,)
        
        # Store in dictionary
        min_max_data[dataset_key] = xr.Dataset({
            "tas_min": (["time"], tas_min.values),
            "tas_max": (["time"], tas_max.values)
        }, coords={"time": time}, 
           attrs={"model_name": model, "scenario": scenario})


In [ ]:
min_max_data_GSAT = {}

for scenario in scenarios:
    for model in LENS_name:
        dataset_key = (model, scenario)
        if dataset_key not in model_datasets:
            print(f"Skipping {dataset_key}, no data available.")
            continue
        
        tas_data = model_datasets[dataset_key]['tas']  # Shape (members, years)
        time = model_datasets[dataset_key]['time'].values  # Get time axis
        
        # Compute min/max across ensemble members
        tas_min = tas_data.min(dim="member")  # Shape (times,)
        tas_max = tas_data.max(dim="member")  # Shape (times,)
        
        # Store in dictionary
        min_max_data_GSAT[dataset_key] = xr.Dataset({
            "tas_min": (["time"], tas_min.values),
            "tas_max": (["time"], tas_max.values)
        }, coords={"time": time}, 
           attrs={"model_name": model, "scenario": scenario})


In [ ]:
min_max_data_GSAT

In [ ]:
min_max_data_AAR = {}

for scenario in scenarios:
    for model in LENS_name:
        dataset_key = (model, scenario)
        if dataset_key not in ARC_ratios_datasets:
            print(f"Skipping {dataset_key}, no data available.")
            continue
        
        tas_data = ARC_ratios_datasets[dataset_key]  # Shape (members, years)
        time = ARC_ratios_datasets[dataset_key]['time'].values  # Get time axis
        
        # Compute min/max across ensemble members
        tas_min = tas_data.min(dim="member")  # Shape (times,)
        tas_max = tas_data.max(dim="member")  # Shape (times,)
        
        # Store in dictionary
        min_max_data_AAR[dataset_key] = xr.Dataset({
            "tas_min": (["time"], tas_min.values),
            "tas_max": (["time"], tas_max.values)
        }, coords={"time": time}, 
           attrs={"model_name": model, "scenario": scenario})
        

In [ ]:
import matplotlib as mpl
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import seaborn as sns
# 'talk' is a context, not a valid style name for seaborn.set_style.
# Use a valid style (e.g. 'white') and apply the 'talk' context for sizing.
sns.set_context('talk', font_scale=0.8)

mpl.rcParams.update({
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
    # "xtick.major.size": 4,
    # "ytick.major.size": 4,
    # "xtick.minor.size": 2,
    # "ytick.minor.size": 2,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "font.size": 10,
})

models     = ['MIROC', 'MIROC-ES2L', 'MPI-ESM-LR', 'CanESM5']
scenarios  = ['ssp119', 'ssp126']

# line style for each model (used for ALL scenarios)
line_set = {
    'ssp119': {'MIROC': 'solid', 'MIROC-ES2L': 'dashed', 'MPI-ESM-LR': 'dashdot', 'CanESM5': (0, (1, 1))},
    'ssp126': {'MIROC': 'solid', 'MIROC-ES2L': 'dashed', 'MPI-ESM-LR': 'dashdot', 'CanESM5': (0, (1, 1))}
}

# SINGLE-model scenario colors
single_color = {
    'ssp119': "#407BD0",  # soft salmon
    'ssp126': "#A32A31",  # magenta
}

# MMLE color + linestyle per scenario
mmle_color = "#501d8a"
mmle_ls = {
    "ssp119": "--",
    "ssp126": ":",
}

def running_mean(da, win=5):
    """Centered running mean on xarray DataArray."""
    return da.rolling(time=win, center=True).mean()

def _concat_over_models(dlist):
    """Concat list of DataArrays along a 'model' dim safely."""
    if len(dlist) == 1:
        return dlist[0].expand_dims(model=[models[0]])
    return xr.concat(dlist, dim='model')

In [ ]:
# add the CO2 concentration increment as a subpanel below the panel a and above panel b, sharing the x-axis with panel b but with its own y-axis on the left
CO2_file = "/work/mh0033/m301036/Atlantic_heat_capacitor_Arctic/Atlantic_capacitor_on_Arctic/docs/data/atmos_co2_concentration/greenhouse_ssp"
SSPs = ["ssp119", "ssp126"]
# %%
# -------------------------------
# Load and process all scenarios
# -------------------------------
co2_list = []
for ssp in SSPs:
    data_path = os.path.join(CO2_file, f"greenhouse_{ssp}.nc")
    ds = xr.open_dataset(data_path, decode_times=False)
    # time is stored as float years (e.g. 2015.0, 2016.0 ...)
    ds = ds.assign_coords(time=ds['time'].values.astype(int))
    # Take global mean if lat/lon dims exist
    co2 = ds.CO2
    if set(['lat', 'lon']).issubset(co2.dims):
        co2 = co2.mean(dim=['lat', 'lon'])
    co2 = co2.sel(time=slice(1980, 2100))
    co2 = co2.assign_coords(scenario=ssp)
    co2_list.append(co2)

# Combine into one DataArray with scenario as a new dimension
CO2_all = xr.concat(co2_list, dim="scenario")

co2_yoy_list = []
for ssp in SSPs:
    co2_ssp = CO2_all.sel(scenario=ssp)
    # year-to-year increment (ppm/yr)
    co2_yoy = co2_ssp.diff("time")
    # align time coordinate to the "current" year (t), not the previous year
    co2_yoy = co2_yoy.assign_coords(time=co2_ssp.time.isel(time=slice(1, None)))
    co2_yoy = co2_yoy.assign_coords(scenario=ssp)
    co2_yoy_list.append(co2_yoy)

CO2_yoy_increment_all = xr.concat(co2_yoy_list, dim="scenario")

In [ ]:
CO2_yoy_increment_all

In [ ]:
# Plot cumulative increments for ssp119 and ssp126
CO2_cumsum_from_1980 = CO2_yoy_increment_all.cumsum("time")

In [ ]:
CO2_cumsum_from_1980

In [ ]:
# ----------------- FIGURE (4 panels: AAR, CO2 cumsum, Arctic SAT, GSAT) -----------------
fig, axes = plt.subplots(
    4, 1, figsize=(12, 22), sharex=True,
    gridspec_kw={"height_ratios": [1, 0.55, 1, 1]}
)
ax_a, ax_co2, ax_b, ax_c = axes

fig.subplots_adjust(left=0.1, right=0.98, top=0.97, bottom=0.18, hspace=0.05)

for ax in axes:
    ax.grid(False)  # type: ignore
    ax.tick_params(
        which="both", direction="out",
        top=False, bottom=True, left=True, right=True, labelright=True
    )

ax_a.tick_params(top=True)

# connect panels visually
ax_a.spines["bottom"].set_visible(False)
ax_co2.spines["top"].set_visible(False)
ax_co2.spines["bottom"].set_visible(False)
ax_b.spines["top"].set_visible(False)
ax_b.spines["bottom"].set_visible(False)
ax_c.spines["top"].set_visible(False)

ax_a.xaxis.set_ticks_position("none")
ax_co2.xaxis.set_ticks_position("none")
ax_b.xaxis.set_ticks_position("none")

ax_a.tick_params(labelbottom=False, labeltop=False)
ax_co2.tick_params(labelbottom=False, labeltop=False)
ax_b.tick_params(labelbottom=False, labeltop=False)
ax_c.tick_params(labeltop=False)

ax_c.set_xlabel("Year")
ax_c.set_xlim(1980, 2100)

# ===== TOP: AAR =====
aar_outlier_thresh = 10.0
for model in models:
    for s in scenarios:
        if (model, s) in min_max_data_AAR:
            t = np.asarray(min_max_data_AAR[(model, s)]["time"])
            lo = np.asarray(min_max_data_AAR[(model, s)]["tas_min"], dtype=float)
            hi = np.asarray(min_max_data_AAR[(model, s)]["tas_max"], dtype=float)
            mask = (np.abs(lo) > aar_outlier_thresh) | (np.abs(hi) > aar_outlier_thresh)
            lo[mask], hi[mask] = np.nan, np.nan
            ax_a.fill_between(t, lo, hi, color=single_color[s], alpha=0.10, zorder=1)

        if (model, s) in ARC_ratio_mean:
            aar_da = ARC_ratio_mean[(model, s)]
            vals = np.asarray(aar_da.values, dtype=float)
            vals[np.abs(vals) > aar_outlier_thresh] = np.nan
            ax_a.plot(
                aar_da["time"].values, vals,
                color=single_color[s], linestyle=line_set[s][model],
                linewidth=3.5, zorder=3
            )

ax_a.set_ylabel("Arctic/GSAT Ratio")
ax_a.set_ylim(-3.5, 7.5)
ax_a.axhline(1.0, lw=2, ls="--", color="k", alpha=0.4)

# ===== PANEL 2: CO2 cumulative sum from 1980 =====
co2_cum_min, co2_cum_max = np.inf, -np.inf
for s in scenarios:
    co2_cum = CO2_cumsum_from_1980.sel(scenario=s)
    t = co2_cum["time"]
    if np.issubdtype(t.dtype, np.datetime64):
        co2_cum = co2_cum.sel(time=slice("1980-01-01", "2100-12-31"))
        years = co2_cum["time"].dt.year.values
    else:
        co2_cum = co2_cum.sel(time=slice(1980, 2100))
        years = co2_cum["time"].values

    y = np.asarray(co2_cum.values, dtype=float)
    ax_co2.plot(
        years, y,
        color=single_color[s], linewidth=3.2, linestyle="-", alpha=1.0, zorder=5,
        label=s.upper()
    )

    finite = y[np.isfinite(y)]
    if finite.size:
        co2_cum_min = min(co2_cum_min, finite.min())
        co2_cum_max = max(co2_cum_max, finite.max())
    
    # Add peak annotations
    peak_value = co2_cum.max().values
    peak_year = co2_cum.time[co2_cum.argmax()].values
    
    # Convert peak_year to numeric if it's datetime64
    if np.issubdtype(type(peak_year), np.datetime64):
        peak_year_num = pd.Timestamp(peak_year).year
    else:
        peak_year_num = float(peak_year)
    
    ax_co2.annotate(
        f"{peak_value:.1f} ppm in {peak_year_num:.0f}",
        xy=(peak_year_num, peak_value),
        xytext=(peak_year_num + 5, peak_value + 10),
        arrowprops=dict(facecolor=single_color[s], shrink=0.05, alpha=0.7),
        fontsize=10,
        color=single_color[s],
        zorder=6
    )
    
    # Add vertical line for peak year
    ax_co2.axvline(
        peak_year_num,
        color=single_color[s],
        linestyle='--',
        linewidth=2,
        alpha=0.6,
        zorder=4
    )

# ax_co2.axhline(0.0, lw=0.8, ls="--", color="k", alpha=0.4)
ax_co2.set_ylabel("Cumulative CO$_2$ change\n(ppm, from 1980)")
if np.isfinite(co2_cum_min) and np.isfinite(co2_cum_max):
    pad = 0.08 * (co2_cum_max - co2_cum_min + 1e-9)
    ax_co2.set_ylim(co2_cum_min - pad, co2_cum_max + pad)

# ===== PANEL 3: Arctic SAT =====
for model in models:
    for s in scenarios:
        if (model, s) in min_max_data:
            t = np.asarray(min_max_data[(model, s)]["time"])
            lo = np.asarray(min_max_data[(model, s)]["tas_min"])
            hi = np.asarray(min_max_data[(model, s)]["tas_max"])
            ax_b.fill_between(t, lo, hi, color=single_color[s], alpha=0.10, zorder=1)

        if model in ARC_ENS_MEAN and s in ARC_ENS_MEAN[model]:
            arc_da = ARC_ENS_MEAN[model][s]
            ax_b.plot(
                arc_da["time"].values, arc_da.values,
                color=single_color[s], linestyle=line_set[s][model],
                linewidth=3.5, zorder=3
            )

ax_b.set_ylabel("Arctic SAT Anomaly (°C)")
ax_b.set_ylim(-3.5, 15)
ax_b.axhline(0.0, lw=2, ls="--", color="k", alpha=0.4)

# ===== PANEL 4: GSAT =====
for model in models:
    for s in scenarios:
        if (model, s) in min_max_data_GSAT:
            t = np.asarray(min_max_data_GSAT[(model, s)]["time"])
            lo = np.asarray(min_max_data_GSAT[(model, s)]["tas_min"])
            hi = np.asarray(min_max_data_GSAT[(model, s)]["tas_max"])
            ax_c.fill_between(t, lo, hi, color=single_color[s], alpha=0.10, zorder=1)

        if (model in model_mean) and (s in model_mean[model]) and ("tas" in model_mean[model][s]):
            gsat_da = model_mean[model][s]["tas"]
            ax_c.plot(
                gsat_da["time"].values, gsat_da.values,
                color=single_color[s], linestyle=line_set[s][model],
                linewidth=3.5, zorder=3
            )

ax_c.set_ylabel("GSAT Anomaly (°C)")
ax_c.set_ylim(-1, 3.5)
ax_c.axhline(0.0, lw=2, ls="--", color="k", alpha=0.4)

# ===== Secondary x-axes on top panel: CO2 concentration =====
co2_119 = CO2_all.sel(scenario="ssp119")
co2_126 = CO2_all.sel(scenario="ssp126")

if np.issubdtype(co2_119["time"].dtype, np.datetime64):
    co2_119 = co2_119.sel(time=slice("1980-01-01", "2100-12-31"))
    co2_126 = co2_126.sel(time=slice("1980-01-01", "2100-12-31"))
    co2_years_119 = co2_119["time"].dt.year.values.astype(float)
    co2_years_126 = co2_126["time"].dt.year.values.astype(float)
else:
    co2_119 = co2_119.sel(time=slice(1980, 2100))
    co2_126 = co2_126.sel(time=slice(1980, 2100))
    co2_years_119 = co2_119["time"].values.astype(float)
    co2_years_126 = co2_126["time"].values.astype(float)

co2_vals_119 = co2_119.values.astype(float)
co2_vals_126 = co2_126.values.astype(float)

def year_to_co2_119(year): return np.interp(year, co2_years_119, co2_vals_119)
def co2_to_year_119(co2): return np.interp(co2, co2_vals_119, co2_years_119)
def year_to_co2_126(year): return np.interp(year, co2_years_126, co2_vals_126)
def co2_to_year_126(co2): return np.interp(co2, co2_vals_126, co2_years_126)

secax_top_119 = ax_a.secondary_xaxis("top", functions=(year_to_co2_119, co2_to_year_119))
secax_top_119.set_xlabel("CO$_2$ (ppm) — SSP119", color=single_color["ssp119"], fontsize=12)
secax_top_119.tick_params(direction="out", labelsize=8, pad=2, colors=single_color["ssp119"], rotation=45)
secax_top_119.spines["top"].set_edgecolor(single_color["ssp119"])

secax_top_126 = ax_a.secondary_xaxis(1.15, functions=(year_to_co2_126, co2_to_year_126))
secax_top_126.set_xlabel("CO$_2$ (ppm) — SSP126", color=single_color["ssp126"], fontsize=12)
secax_top_126.tick_params(direction="out", labelsize=8, pad=2, colors=single_color["ssp126"], rotation=45)
secax_top_126.spines["top"].set_edgecolor(single_color["ssp126"])

# panel tags
for ax, tag in zip(axes, ["a", "b", "c", "d"]):
    ax.text(0.97, 0.07, tag, transform=ax.transAxes,
            fontweight="bold", ha="left", va="top", fontsize=22)

# legend
# Define member counts for each model
member_counts = {
    "CanESM5": 50,
    "MIROC": 50,
    "MIROC-ES2L": 10,
    "MPI-ESM1-2-LR": 50,
    "MPI-ESM-LR": 50  
}
model_name_mapping = {
    "MPI-ESM1-2-LR": "MPI-ESM-LR",
    "MIROC": "MIROC6",
    "MIROC-ES2L": "MIROC-ES2L",
    "CanESM5": "CanESM5"
}

# When creating legend entries, add the count
# Build model legend entries (fixes: models_name_mapping -> model_name_mapping, and m -> model)
model_handles = []
for model in models:
    display_name = model_display_names.get(model, model_name_mapping.get(model, model))
    n_members = member_counts.get(model, "N/A")
    model_handles.append(
        Line2D(
            [0], [0],
            color="0.3",
            lw=2.0,
            linestyle=line_set["ssp119"].get(model, "-"),
            label=f"{display_name} ({n_members})"
        )
    )

scenario_handles = [
    Line2D([0], [0], color=single_color["ssp119"], lw=0, marker="s", markersize=8, label="SSP1-1.9"),
    Line2D([0], [0], color=single_color["ssp126"], lw=0, marker="s", markersize=8, label="SSP1-2.6"),
]
fig.legend(
    handles=model_handles + scenario_handles,
    loc="lower center", bbox_to_anchor=(0.5, 0.1), ncol=4, frameon=False
)
dir_out_fig = "/work/mh0033/m301036/Atlantic_heat_capacitor_Arctic/Atlantic_capacitor_on_Arctic/docs/figs/SI_Figs"
plt.savefig(f"{dir_out_fig}/Extended_Data_Fig1-SSPs-ARCSAT-GSAT-AAR-CO2cumsum-1980-2100.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{dir_out_fig}/Extended_Data_Fig1-SSPs-ARCSAT-GSAT-AAR-CO2cumsum-1980-2100.pdf", dpi=300, bbox_inches="tight")
plt.show()